In [1]:
#Assignmentday2

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("UnionExample").getOrCreate()


data1 = [("apple", 3, 5),
         ("banana", 1, 10),
         ("orange", 2, 8)]
columns1 = ["Name", "Col_1", "Col_2"]

df1 = spark.createDataFrame(data1, columns1)  #firstdf


data2 = [("apple", 3, 5),
         ("banana", 1, 15),
         ("grape", 4, 6)]
columns2 = ["Name", "Col_1", "Col_3"]

df2 = spark.createDataFrame(data2, columns2)  #secondf

# Renaming so that both df will look same or same schema
df2_renamed = df2.withColumnRenamed("Col_3", "Col_2")

result= df1.union(df2_renamed)
result.show()

+------+-----+-----+
|  Name|Col_1|Col_2|
+------+-----+-----+
| apple|    3|    5|
|banana|    1|   10|
|orange|    2|    8|
| apple|    3|    5|
|banana|    1|   15|
| grape|    4|    6|
+------+-----+-----+



In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("ExtractEmails").getOrCreate()

data = [
    ("buying books at amazom.com",),
    ("rameses@egypt.com",),
    ("matt@t.co",),
    ("narendra@modi.com",)
]
columns = ["value"]

df = spark.createDataFrame(data, columns)

# to know which contain @ or looks like actual email
filtered_df = df.filter(col("value").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"))

filtered_df.show(truncate=False) #fulltext


+-----------------+
|value            |
+-----------------+
|rameses@egypt.com|
|matt@t.co        |
|narendra@modi.com|
+-----------------+



In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as _sum

spark = SparkSession.builder.appName("PivotExample").getOrCreate()

data = [
    (2021, 1, "US", 5000),
    (2021, 1, "EU", 4000),
    (2021, 2, "US", 5500),
    (2021, 2, "EU", 4500),
    (2021, 3, "US", 6000),
    (2021, 3, "EU", 5000),
    (2021, 4, "US", 7000),
    (2021, 4, "EU", 6000)
]

columns = ["year", "quarter", "region", "revenue"]

df = spark.createDataFrame(data, columns)

# Pivot region colum
pivot_df = (
    df.groupBy("year", "quarter")
      .pivot("region")
      .agg(_sum("revenue"))
      .orderBy("year", "quarter")
)

pivot_df.show()


+----+-------+----+----+
|year|quarter|  EU|  US|
+----+-------+----+----+
|2021|      1|4000|5000|
|2021|      2|4500|5500|
|2021|      3|5000|6000|
|2021|      4|6000|7000|
+----+-------+----+----+



In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, split, explode, count, lit, first

spark = SparkSession.builder.appName("ReplaceMissingSpaces").getOrCreate()

data = [("dbc deb abed gade",)]
columns = ["string"]

df = spark.createDataFrame(data, columns)

# Split string into characters and count frequency (excluding spaces)
char_df = (
    df.withColumn("char", explode(split(regexp_replace(col("string"), " ", ""), "")))
      .groupBy("char")
      .agg(count("*").alias("freq"))
      .orderBy("freq")
)

# Get least frequent character
least_freq_char = char_df.limit(1).collect()[0]["char"]

# Replace spaces in original string with that character
result = df.withColumn("modified_string", regexp_replace(col("string"), " ", least_freq_char))

result.show(truncate=False)


+-----------------+-----------------+
|string           |modified_string  |
+-----------------+-----------------+
|dbc deb abed gade|dbcgdebgabedggade|
+-----------------+-----------------+



In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, when

spark = SparkSession.builder.appName("MissingValuesCheck").getOrCreate()

data = [
    ("A", 1, None),
    ("B", None, 123),
    ("B", 3, 456),
    ("D", None, None)
]
columns = ["Name", "Value", "id"]

df = spark.createDataFrame(data, columns)

# Count missing (null) values in each column
missing_counts = df.select([
    _sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns
]).collect()[0].asDict()

# Check if any column has missing values
has_missing = any(v > 0 for v in missing_counts.values())

print(has_missing)
print(missing_counts)


True
{'Name': 0, 'Value': 2, 'id': 2}


In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

spark = SparkSession.builder.appName("NthRowFilter").getOrCreate()

data = [
    ("Alice", 1),
    ("Bob", 2),
    ("Charlie", 3),
    ("Dave", 4),
    ("Eve", 5),
    ("Frank", 6),
    ("Grace", 7),
    ("Hannah", 8),
    ("Igor", 9),
    ("Jack", 10)
]
columns = ["Name", "Number"]

df = spark.createDataFrame(data, columns)

# Add row number
windowSpec = Window.orderBy("Number")
df_with_rn = df.withColumn("rn", row_number().over(windowSpec))

# Filter every nth row (eg every 5th row)
n = 5
filtered_df = df_with_rn.filter(col("rn") % n == 0)
filtered_df.show()


+----+------+---+
|Name|Number| rn|
+----+------+---+
| Eve|     5|  5|
|Jack|    10| 10|
+----+------+---+



In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = SparkSession.builder.appName("ColumnMatchCheck").getOrCreate()

data = [
    ("John", "John"),
    ("Lily", "Lucy"),
    ("Sam", "Sam"),
    ("Lucy", "Lily")
]
columns = ["Name1", "Name2"]

df = spark.createDataFrame(data, columns)

# Compare columns and create a new 'Match' column
df_match = df.withColumn("Match", when(col("Name1") == col("Name2"), True).otherwise(False))
df_match.show()


+-----+-----+-----+
|Name1|Name2|Match|
+-----+-----+-----+
| John| John| true|
| Lily| Lucy|false|
|  Sam|  Sam| true|
| Lucy| Lily|false|
+-----+-----+-----+



In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, lead, col

spark = SparkSession.builder.appName("LagLeadExample").getOrCreate()

data = [
    ("2023-01-01", "Store1", 100),
    ("2023-01-02", "Store1", 150),
    ("2023-01-03", "Store1", 200),
    ("2023-01-04", "Store1", 250),
    ("2023-01-05", "Store1", 300),
    ("2023-01-01", "Store2", 50),
    ("2023-01-02", "Store2", 60),
    ("2023-01-03", "Store2", 80),
    ("2023-01-04", "Store2", 90),
    ("2023-01-05", "Store2", 120)
]
columns = ["Date", "Store", "Sales"]

df = spark.createDataFrame(data, columns)

# Define a window partitioned by Store and ordered by Date
windowSpec = Window.partitionBy("Store").orderBy("Date")

# Create Lag and Lead columns
df_lag_lead = df.withColumn("Lag_Sales", lag("Sales", 1).over(windowSpec)) \
                .withColumn("Lead_Sales", lead("Sales", 1).over(windowSpec))

df_lag_lead.show()


+----------+------+-----+---------+----------+
|      Date| Store|Sales|Lag_Sales|Lead_Sales|
+----------+------+-----+---------+----------+
|2023-01-01|Store1|  100|     NULL|       150|
|2023-01-02|Store1|  150|      100|       200|
|2023-01-03|Store1|  200|      150|       250|
|2023-01-04|Store1|  250|      200|       300|
|2023-01-05|Store1|  300|      250|      NULL|
|2023-01-01|Store2|   50|     NULL|        60|
|2023-01-02|Store2|   60|       50|        80|
|2023-01-03|Store2|   80|       60|        90|
|2023-01-04|Store2|   90|       80|       120|
|2023-01-05|Store2|  120|       90|      NULL|
+----------+------+-----+---------+----------+



In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("UniqueValueFrequency").getOrCreate()

data = [
    (1, 2, 3),
    (2, 3, 4),
    (1, 2, 3),
    (4, 5, 6),
    (2, 3, 4)
]
columns = ["Column1", "Column2", "Column3"]

df = spark.createDataFrame(data, columns)

# Combine all columns into one (union all columns)
stacked_df = (
    df.select(col("Column1").alias("single_column"))
    .union(df.select(col("Column2").alias("single_column")))
    .union(df.select(col("Column3").alias("single_column")))
)

# Group by the single column and count frequency
freq_df = stacked_df.groupBy("single_column").count().orderBy(col("count").desc())
freq_df.show()


+-------------+-----+
|single_column|count|
+-------------+-----+
|            2|    4|
|            3|    4|
|            4|    3|
|            1|    2|
|            5|    1|
|            6|    1|
+-------------+-----+



In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id, col

spark = SparkSession.builder.appName("ReverseRows").getOrCreate()

data = [
    (1, 2, 3, 4),
    (2, 3, 4, 5),
    (3, 4, 5, 6),
    (4, 5, 6, 7)
]
columns = ["col_1", "col_2", "col_3", "col_4"]

df = spark.createDataFrame(data, columns)

# Add a monotonically increasing index
df_with_index = df.withColumn("row_id", monotonically_increasing_id())

# Order by row_id descending to reverse the order
reversed_df = df_with_index.orderBy(col("row_id").desc()).drop("row_id")
reversed_df.show()


+-----+-----+-----+-----+
|col_1|col_2|col_3|col_4|
+-----+-----+-----+-----+
|    4|    5|    6|    7|
|    3|    4|    5|    6|
|    2|    3|    4|    5|
|    1|    2|    3|    4|
+-----+-----+-----+-----+

